In [2]:
# Importul bibliotecilor esențiale pentru ML și analiza audio

import librosa        # Procesarea semnalului audio (MFCC)
import soundfile      # Citirea fișierelor audio
import numpy as np    # Operații matematice
import glob           # Pentru a găsi toate fișierele audio în foldere
import os             # Pentru lucrul cu căi de fișiere

# Biblioteci pentru Machine Learning 
from sklearn.model_selection import train_test_split # Split în train/test
from sklearn.neural_network import MLPClassifier     # Modelul de rețea neuronală
from sklearn.metrics import accuracy_score           # Evaluarea performanței

In [ ]:
# Functia de extragere a caracteisticilor MFFC (Cât de ascuțită sau groasă este vocea.)  si Mel-Spectogram din fisierele audio (Cât de tare sau încet se vorbește (intensitatea).)

#transform sunetul brut într-o amprentă vocală digitală. Această amprentă este compusă din Frecvență (tonul vocii) și Volum (intensitatea)
def extract_feature(file_name, mfcc=True, mel=True):
    try:
        with soundfile.SoundFile(file_name) as sound_file:
            X = sound_file.read(dtype="float32")
            sample_rate = sound_file.samplerate
            
            # Verificăm dacă fișierul este prea scurt 
            if len(X) < 2048:
                return None
            
            result = np.array([])
            
            # 1. MFCC - Rămâne activ
            if mfcc:
                mfccs = np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
                result = np.hstack((result, mfccs))
            
            # 2. Mel Spectrogram - Rămâne activ
            if mel:
                mel_spec = np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T, axis=0)
                result = np.hstack((result, mel_spec))
                
            return result
    except Exception as e:
        return None

In [4]:
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split

#  DEFINIREA EMOȚIILOR 
emotions = {
    '01':'neutral',
    '02':'calm',
    '03':'happy',
    '04':'sad',
    '05':'angry',
    '06':'fearful',
    '07':'disgust',
    '08':'surprised'
}

# Aici alegem emotiile
observed_emotions = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

#  FUNCȚIA DE ÎNCĂRCARE DATE 
def load_data(test_size=0.25):
    X, y = [], []
    count = 0
    
    # Construim calea pentru Windows
    cale_cautare = os.path.join("actors_voices", "Actor_*", "*.wav")
    files = glob.glob(cale_cautare)
    
    print(f"DEBUG: Am găsit în total {len(files)} fișiere .wav")

    for file in files:
        file_name = os.path.basename(file)
        parts = file_name.split("-")
        
        if len(parts) >= 3:
            emotion_code = parts[2]
            
            # Verificăm dacă codul există în dicționarul nostru
            if emotion_code in emotions:
                emotion = emotions[emotion_code]
                
                # Verificăm dacă emoția este una din cele 8 pe care le vrem
                if emotion in observed_emotions:
                    # Folosim funcția de extracție creată anterior
                    feature = extract_feature(file, mfcc=True, mel=True)
                    if feature is not None:
                        X.append(feature)
                        y.append(emotion)
                        count += 1
            
    print(f" Am procesat cu succes {count} fișiere pentru cele 8 emoții alese. ")
    
    if count == 0:
        print("EROARE: Nu am putut procesa niciun fișier. Verifică folderul 'actors_voices'!")
        return None, None, None, None
        
    return train_test_split(np.array(X), y, test_size=test_size, random_state=9)

#  PASUL C: RULARE 
X_train, X_test, y_train, y_test = load_data(test_size=0.25)

DEBUG: Am găsit în total 1440 fișiere .wav


c:\Users\Maria\AppData\Local\Programs\Python\Python310\lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=2
  warnings.warn(


 Am procesat cu succes 1435 fișiere pentru cele 8 emoții alese. 


In [5]:
# Model pentru 8 emoții
model = MLPClassifier(
    alpha=0.001,                # Penalizare mai mică (permite modelului să fie mai complex)
    batch_size=128,             # Procesăm datele în bucati mai mici
    hidden_layer_sizes=(1024, 512, 256), # Trei straturi
    learning_rate='adaptive', 
    max_iter=1000,              # Mai multe încercări de învățare
    random_state=9,
    verbose=True                # Aceasta ne va arăta progresul în timp real 
)

model.fit(X_train, y_train)

# Evaluare
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_true=y_test, y_pred=y_pred)
print("\n" + "="*40)
print(f"ACURATEȚE: {accuracy*100:.2f}%")
print("="*40)

Iteration 1, loss = 24.19180942
Iteration 2, loss = 14.30029418
Iteration 3, loss = 6.28778580
Iteration 4, loss = 3.50578409
Iteration 5, loss = 2.34769184
Iteration 6, loss = 2.11105205
Iteration 7, loss = 1.93020753
Iteration 8, loss = 1.75168090
Iteration 9, loss = 1.66564035
Iteration 10, loss = 1.70085362
Iteration 11, loss = 1.73287903
Iteration 12, loss = 1.62993535
Iteration 13, loss = 1.64356906
Iteration 14, loss = 1.59372553
Iteration 15, loss = 1.70243133
Iteration 16, loss = 1.61817048
Iteration 17, loss = 1.84004141
Iteration 18, loss = 1.77453484
Iteration 19, loss = 1.61766115
Iteration 20, loss = 1.46447669
Iteration 21, loss = 1.40443116
Iteration 22, loss = 1.39390626
Iteration 23, loss = 1.36266774
Iteration 24, loss = 1.34055753
Iteration 25, loss = 1.37901315
Iteration 26, loss = 1.45743183
Iteration 27, loss = 1.37690482
Iteration 28, loss = 1.34454589
Iteration 29, loss = 1.33845984
Iteration 30, loss = 1.46071922
Iteration 31, loss = 1.36383103
Iteration 32, l

In [6]:
# Functia de recunoastere a emotiei
def recunoaste_emotia(cale_fisier):
    # 1. Aplicăm procesarea pe fișierul nou
    caracteristici = extract_feature(cale_fisier, mfcc=True, mel=True)
    
    if caracteristici is not None:
        # 2. Pregătim datele pentru model
        caracteristici = caracteristici.reshape(1, -1)
        
        # 3. Întrebăm creierul antrenat ce crede
        predictie = model.predict(caracteristici)
        
        return predictie[0]
    else:
        return "Nu am putut procesa fisierul."

# TESTARE

fisier_nou = "actors_voices/Actor_05/03-01-03-01-01-01-05.wav" 

rezultat = recunoaste_emotia(fisier_nou)

print(f" The emotion is: {rezultat.upper()}")

 The emotion is: HAPPY


In [7]:
import joblib

# Salvează "creierul" într-un fișier numit 'model_emotii_ser.pkl'
joblib.dump(model, 'model_emotii_ser.pkl')

print(" Modelul a fost salvat!")

 Modelul a fost salvat!


In [8]:
import gradio as gr

def predict_emotion_ui(audio_file):
    try:
        # Folosim doar ce am antrenat: mfcc=True, mel=True
        
        features = extract_feature(audio_file, mfcc=True, mel=True)
        
        if features is not None:
            features = features.reshape(1, -1)
            prediction = model.predict(features)
            emotia = prediction[0]
            
            # Mapare vizuală
            emoji_map = {
                "happy": "😊 Happy", "sad": "😢 Sad", "angry": "😡 Angry", 
                "calm": "😌 Calm", "neutral": "😐 Neutral", "surprised": "😲 Surprised",
                "fearful": "😨 Fearful", "disgust": "🤢 Disgust"
            }
            return emoji_map.get(emotia, emotia.upper())
        else:
            return " Could not process sound."
    except Exception as e:
        return f" Eroare: {str(e)}"

# Interfață curată
with gr.Blocks(title="AI Emotion Detector") as demo:
    gr.Markdown("#  Voice Emotion Analyzer")
    gr.Markdown(f"Trained model")
    
    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(type="filepath", label=" Load a .wav file from the dataset")
            submit_btn = gr.Button("🔍 Detect Emotion", variant="primary")
        with gr.Column():
            output_text = gr.Label(label="Result")

    submit_btn.click(fn=predict_emotion_ui, inputs=audio_input, outputs=output_text)

demo.launch()

c:\Users\Maria\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
